# Типы памяти в ИИ-агентах и инструменты управления памятью

> Внимание! Материал ноутбука подходит для работы в Google Colaboratory. Мы не можем гарантировать стабильную работу кода на личных устройствах и на других системах виртуализации.

**Вы научитесь:**
- различать, какие ошибки агента лечатся окном контекста, какие - записью факта наружу, а какие - сохранением способа решения
- ставить точки записи и чтения памяти в цикле агента и назначать каждой записи срок жизни
- собирать промпт так, чтобы запрос попадал в кэш контекста, и замерять, где попали
- подключать Mem0 к агенту на GigaChat, помечать, что именно вы сохранили, и доставать из памяти только то, что относится к вопросу
- считать, во сколько обходится память, - и решать, стоит ли она этих денег в вашей задаче

## Введение

Первые четыре занятия агент жил внутри одного обращения: получил задачу, сходил инструментами, ответил. На занятии 4 таких агентов стало много, они разъехались по очередям и шардам - но каждый по-прежнему начинал с чистого листа.

В продукте так не бывает. Пользователь возвращается пишет «сделай как в прошлый раз». Оператор ждет, что ассистент помнит тариф и данные клиента. Инженер надеется, что агент не будет третий раз подряд перебирать те же тупиковые ходы.

**Память агента** - это все, что переживает границу одного вызова модели. Дальше разберем, из чего она состоит, где в цикле агента стоят точки записи и чтения, сколько это стоит в рублях и что за нас уже сделано в готовых библиотеках - **Mem0** и **Mirix**.

Работать будем на ассистенте поддержки облачного сервиса: он отвечает на обращения клиентов, у него есть доступ к данным о проектах и квотах.

## Установка зависимостей

Нам понадобятся:
- `mem0ai` - готовый слой памяти: извлечение записей из диалога, хранение, поиск;
- `langchain` - **полный** пакет, не только `langchain-core`: обертки Mem0 импортируют `langchain.chat_models.base` и `langchain.embeddings.base`, без него провайдер `langchain` не поднимется;
- `langchain-gigachat` - модель и эмбеддинги.

Версию Mem0 закрепляем жестко. Библиотека молодая и ломает API в минорных выпусках: код из статей и туториалов годичной давности на текущей версии просто не работает - ниже увидим, почему.

In [ ]:
%pip install -q mem0ai==2.0.18 "langchain>=1.0,<2" langchain-gigachat

In [1]:
import os
from langchain_gigachat import GigaChat, GigaChatEmbeddings

from dotenv import load_dotenv
load_dotenv()

llm = GigaChat(
    credentials=os.environ["GIGACHAT_CREDENTIALS"],
    scope="GIGACHAT_API_B2B",
    model="GigaChat-2-Max",
    verify_ssl_certs=False,
    timeout=120,
)
emb = GigaChatEmbeddings(
    credentials=os.environ["GIGACHAT_CREDENTIALS"],
    scope="GIGACHAT_API_B2B",
    verify_ssl_certs=False,
)
print("модель и эмбеддинги подключены")

модель и эмбеддинги подключены


## Три отказа агента без памяти

Начнем не с классификации, а с трех поломок. Они выглядят одинаково («агент забыл»), но лечатся совершенно разными способами - и в этом основная тема занятия.

Вот фрагмент переписки с клиентом. Обращение длинное, как это обычно и бывает.

In [2]:
DIALOG = [
    ("user",      "Здравствуйте. У меня проект PRJ-77, воркеры падают с ошибкой квоты."),
    ("assistant", "Здравствуйте! Уточните, пожалуйста, когда началось."),
    ("user",      "Сегодня утром, часов с девяти по Владивостоку."),
    ("assistant", "Спасибо. Проверяю загрузку кластера."),
    ("user",      "И еще: пишите мне, пожалуйста, без лишних вежливых оборотов, по делу."),
    ("assistant", "Понял, буду коротко."),
    ("user",      "Что там по загрузке?"),
    ("assistant", "Кластер загружен на 60 процентов, это норма."),
    ("user",      "Тогда почему падает?"),
    ("assistant", "Смотрю лимиты по вашему тарифу."),
    ("user",      "Тариф Pro, оплачен до декабря."),
    ("assistant", "Вижу. Проверяю квоту на параллельные задачи."),
    ("user",      "Ну и что в итоге с моими воркерами?"),
]

WINDOW = 6  # столько последних реплик влезает в бюджет контекста

window = DIALOG[-WINDOW:]
print("В модель уходит только это:\n")
for role, text in window:
    print(f"  {role:9} | {text}")

fell_out = [t for r, t in DIALOG[:-WINDOW]]
print("\nЗа границей окна осталось:", len(fell_out), "реплики")
print("Например:", fell_out[0])

В модель уходит только это:

  assistant | Кластер загружен на 60 процентов, это норма.
  user      | Тогда почему падает?
  assistant | Смотрю лимиты по вашему тарифу.
  user      | Тариф Pro, оплачен до декабря.
  assistant | Вижу. Проверяю квоту на параллельные задачи.
  user      | Ну и что в итоге с моими воркерами?

За границей окна осталось: 7 реплики
Например: Здравствуйте. У меня проект PRJ-77, воркеры падают с ошибкой квоты.


**Что здесь видно:**
- номер проекта `PRJ-77`, часовой пояс и просьба писать короче произнесены в начале - и выпали из окна;
- модель не «забыла» их: этого текста физически нет в том, что ей отправили. Спрашивать «почему агент забыл» бессмысленно, надо спрашивать «что мы отправили LLM».

Это **первый отказ: потеря информации внутри одного диалога**. Он про бюджет контекста.

Второй отказ виден еще проще. Клиент возвращается на следующий день - и начинается новая сессия.

In [3]:
SYSTEM = "Ты ассистент поддержки облачного сервиса. Отвечай кратко и по делу."

session_2 = [
    ("system", SYSTEM),
    ("user",   "Привет, вчера чинили падение воркеров. Продолжим?"),
]

print("Все, что знает агент во второй сессии:\n")
for role, text in session_2:
    print(f"  {role:9} | {text}")

print("\nНи номера проекта, ни тарифа, ни просьбы писать коротко здесь нет.")
print("Это не потеря по дороге - это чистый лист.")

Все, что знает агент во второй сессии:

  system    | Ты ассистент поддержки облачного сервиса. Отвечай кратко и по делу.
  user      | Привет, вчера чинили падение воркеров. Продолжим?

Ни номера проекта, ни тарифа, ни просьбы писать коротко здесь нет.
Это не потеря по дороге - это чистый лист.


**Второй отказ: потеря между сессиями.** Лечится он не окном контекста - окно тут пустое по определению, - а тем, что важное выносится в память до завершения первой сессии.

Третий отказ хуже всех, и, кстати, говорят о нем редко. Вчера у другого клиента была ровно такая же поломка, и агент нашел правильный рабочий порядок действий - через три неудачные попытки.

In [4]:
# журнал вчерашнего обращения другого клиента (такой лежит у всех, кто пишет трейсы)
YESTERDAY = [
    {"step": 1, "tool": "restart_workers",  "result": "ошибка повторилась"},
    {"step": 2, "tool": "check_cluster",    "result": "загрузка в норме, дело не в ней"},
    {"step": 3, "tool": "check_billing",    "result": "оплата прошла, дело не в ней"},
    {"step": 4, "tool": "check_quota",      "result": "лимит параллельных задач исчерпан"},
    {"step": 5, "tool": "raise_quota",      "result": "помогло"},
]

useful = [s for s in YESTERDAY if "не в ней" not in s["result"] and "повторилась" not in s["result"]]
print("Шагов сделано:", len(YESTERDAY), "| из них полезных:", len(useful))
print("Рабочий порядок:", " -> ".join(s["tool"] for s in useful))
print()
print("Этот вывод нигде не сохранен: журнал есть, а понимания «как надо» - нет.")
print("Сегодня агент снова начнет с restart_workers.")

Шагов сделано: 5 | из них полезных: 2
Рабочий порядок: check_quota -> raise_quota

Этот вывод нигде не сохранен: журнал есть, а понимания «как надо» - нет.
Сегодня агент снова начнет с restart_workers.


**Третий отказ: потеря между задачами.** Тут теряются не факты о пользователе, а **способ решения**. Ни окно контекста, ни профиль клиента его не лечат.

Три отказа - три разных механизма:

| Отказ | Где теряется | Как исправить |
|---|---|---|
| Внутри диалога | не влезло в окно | управление окном и кэш контекста |
| Между сессиями | сессия кончилась | запись факта во внешнее хранилище |
| Между задачами | опыт нигде не сохранился | запись способа решения |

Убедимся на живой модели, что первый отказ выглядит именно так, как мы описали.

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Собираем ровно то, что уйдет модели: системная инструкция плюс урезанное окно диалога.
ROLES = {"system": SystemMessage, "user": HumanMessage, "assistant": AIMessage}
messages = [ROLES[r](t) for r, t in ([("system", SYSTEM)] + list(window))]

# На какой вопрос отвечаем и что из сказанного клиентом до модели не доехало
question = window[-1][1]
sent = SYSTEM + " " + " ".join(text for _, text in window)

print("вопрос клиента:", question)
print("а вот чего в отправленном нет:")
for fact in ["PRJ-77", "Владивосток", "без лишних вежливых оборотов"]:
    print(f"   {fact:30} есть в промпте: {fact in sent}")

answer = llm.invoke(messages)
print("\nответ агента:", answer.content)

вопрос клиента: Ну и что в итоге с моими воркерами?
а вот чего в отправленном нет:
   PRJ-77                         есть в промпте: False
   Владивосток                    есть в промпте: False
   без лишних вежливых оборотов   есть в промпте: False

ответ агента: Обнаружено превышение лимита параллельных процессов. Рекомендую оптимизировать нагрузку или расширить пакет ресурсов тарифа Pro.


Посмотрите на ответ внимательно. Агент **не переспросил** номер проекта - он уверенно поставил диагноз и порекомендовал сменить тариф, хотя ни номера проекта, ни причины сбоя в промпте не было. Это важнее, чем кажется: отсутствие контекста не делает агента осторожным, оно делает его выдумщиком. Претензии при этом не к модели - ей просто не дали того, о чем спрашивают.

Дальше по очереди разберем каждый из способов исправления разобранных ошибок. А чтобы говорить о них точно, сначала нужен словарь.

## Память по типу информации

Первое деление - **по типу информации**. Оффтопный факт: оно пришло из когнитивной психологии, но для нас важно другое: у каждого типа своя форма записи и свои момент чтения.

- **Семантическая** - устойчивые факты о мире и о пользователе: тариф, часовой пояс, предпочтения. Отвечает на вопрос «что истинно».
- **Эпизодическая** - что произошло и когда: обращение такое-то закрыто тогда-то. Отвечает на вопрос «что было».
- **Процедурная** - как решать задачу: последовательность действий, которая сработала. Отвечает на вопрос «как надо».

Разница видна на записях. Возьмем один и тот же вчерашний случай и запишем его тремя способами.

In [ ]:
from dataclasses import dataclass, field, asdict
from datetime import date

@dataclass
class Record:
    kind: str            # semantic | episodic | procedural
    text: str
    subject: str         # к кому или к чему относится
    happened_at: str = ""    # только для эпизода
    ttl_days: int | None = None   # срок жизни, None - бессрочно

records = [
    Record("semantic", "Тариф Pro, оплачен до декабря", subject="client:42", ttl_days=120),
    Record("episodic", "Падение воркеров из-за исчерпанной квоты, помогло повышение лимита",
           subject="client:42", happened_at="2026-08-12", ttl_days=None),
    Record("procedural", "При падении воркеров с ошибкой квоты: check_quota -> raise_quota",
           subject="скрипт: падение воркеров", ttl_days=None),
]

for r in records:
    print(f"{r.kind:11} | к кому: {r.subject:28} | {r.text}")

**Разбор вывода:**
- у семантической записи есть срок годности: тариф меняется, и через сто двадцать дней факт стоит перепроверить;
- у эпизода срока нет, но есть дата: он не устаревает, он просто становится более отдаленным во времени;
- у процедурной записи нет владельца-клиента: она относится к типу поломки и полезна всем клиентам сразу.

Одно это уже задает разное поведение хранилища. Дальше разрежем ту же память по другому признаку.

## Память по длительности

Второе деление - **по длительности**:

- **кратковременная** - то, что живет внутри одного хода: промежуточные мысли, результат вызова инструмента;
- **рабочая** - то, что агент держит в процессе решения задачи: текущее обращение, собранные по ходу факты;
- **долговременная** - то, что переживает сессию.

Здесь нужна честная оговорка. **В работающем агенте кратковременная и рабочая память - это одно и то же место: список сообщений в окне контекста.** Разделение между ними существует в психологии, а у нас обе живут в одной структуре данных и умирают одновременно - когда сессия закрылась или когда реплики выпали из окна.

Практический смысл этого деления ровно один: провести границу между тем, что исчезнет само, и тем, что потребуется сохранить. Граница проходит между рабочей и долговременной памятью, и проходит она через **точку записи** - момент, когда мы что-то вынимаем из диалога и кладем в хранилище.

## Память по исполнению

Третье деление - **по исполнению**, то есть по тому, где физически лежит знание:

- **параметрическая** - в весах модели. Туда попадает то, на чем модель обучали;
- **внешняя** - в хранилище рядом с агентом: база, файл, векторный индекс;
- **декларативная** - записанная словами в промпте: системная инструкция, вставленный профиль клиента.

И снова оговорка, без которой деление бесполезно. **Параметрической памятью вы не управляете**, если не дообучаете модель, - а в задачах этого курса не дообучаете. Знание внутри весов нельзя обновить, нельзя удалить по требованию клиента и нельзя проверить на актуальность.

Поэтому в реальном проекте выбор всегда между двумя оставшимися: положить знание **во внешнее хранилище** и доставать по надобности - или держать его **прямо в промпте** постоянно. Первое дешевле по токенам и сложнее в сопровождении, второе наоборот.

## Где эта классификация не работает

Три деления выглядят как три оси, по которым можно разложить любую запись. На практике так не выходит, и лучше знать об этом заранее:

1. **Оси не независимы.** Процедурная запись почти всегда долговременная и внешняя - других сочетаний просто не встречается. Из двадцати семи клеток пересечений заполнены единицы.
2. **Часть клеток не про нас.** Параметрическая память не управляется, кратковременная не отделима от рабочей.
3. **Классификация не говорит, что делать.** Из того, что запись «эпизодическая», не следует ни где ее хранить, ни когда писать, ни когда удалять.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

QUESTION = "Какой тариф у клиента с проектом PRJ-77 и до какого числа он оплачен?"

print("вопрос без контекста:")
print("  ->", llm.invoke([HumanMessage(QUESTION)]).content.strip()[:180])

FACT = "Клиент проекта PRJ-77: тариф Pro, оплачен до декабря 2026 года."
print("\nтот же вопрос, факт положен в промпт:")
print("  ->", llm.invoke([SystemMessage(FACT), HumanMessage(QUESTION)]).content.strip()[:180])

**Пояснения к результату:**
- в первый раз модель отвечает, что таких данных у нее нет. Это лучший из
  возможных исходов, но не гарантированный: в начале занятия, когда контекста
  не хватало частично, а не полностью, та же модель уверенно поставила диагноз
  и посоветовала сменить тариф;
- во второй раз ответ верный, и изменилось ровно одно - факт оказался в промпте.
  Ни настройкой, ни промптом системы этого нельзя добиться от весов: параметрическую
  память не обновить, не удалить по требованию клиента и не проверить на актуальность;
- отсюда вывод раздела: из трех видов памяти по исполнению в проекте вы управляете
  двумя - внешним хранилищем и текстом в промпте. Все занятие дальше про то, как
  выбирать между ними и что именно класть в промпт на каждом ходу.

## Четыре решения, которые на самом деле принимает инженер

Что бы вы ни выбрали - самописную память, Mem0 или Mirix, - проектирование памяти сводится к четырем вопросам.

| Вопрос | Что решаем | Пример нашего ассистента |
|---|---|---|
| **Что записать** | какие фрагменты диалога достойны хранения | тариф и часовой пояс - да, «спасибо, помогло» - нет |
| **Когда записать** | точка записи в цикле агента | после ответа клиента, после успешного вызова инструмента, в конце сессии |
| **Где хранить** | хранилище и форма записи | внешняя база с полями «тип, владелец, срок» |
| **Что подставить** | точка чтения и бюджет | три самые близкие записи по смыслу, не больше 500 токенов |

Дальше занятие идет ровно по этим четырем вопросам. Начнем с самого дешевого варианта - когда наружу вообще ничего писать не надо.

## Память и RAG: в чем разница

Вопрос возникает сразу, потому что механика похожа: и там, и там эмбеддинги, векторный поиск и вставка найденного в промпт. Разница не в поиске, а в **том, кто пишет**:

- в RAG корпус готовят заранее и снаружи: документы, инструкции, база знаний. Агент только читает;
- в памяти записи порождает сам агент по ходу работы. Он и пишет, и читает.

Отсюда все различия в проблемах. У RAG нет вопроса «что считать фактом» - фрагменты уже даны. У памяти нет вопроса «как разбить документ на фрагменты» - зато есть вопрос, что делать, когда вчерашний факт противоречит сегодняшнему.

Устройство хранилищ и гибридный поиск подробно разбираются на занятии 7, разрешение противоречий - на занятии 8. Нам сейчас важна только граница: дальше мы говорим про то, что агент записал сам.

## Кэш контекста: что кэшируется и что это дает

Самый дешевый способ не терять контекст - не выносить его никуда, а просто продолжать передавать модели. Проблема очевидна: за длинный промпт платишь на каждом ходу заново.

Ровно это и решает **кэш контекста** (prompt cache). У GigaChat он устроен так:

- запрос помечается заголовком `X-Session-ID` - произвольной строкой, которую вы придумываете сами (если не передать, сервер подставит случайную, и кэш не попадет);
- сервер ищет в кэше запрос с тем же идентификатором и **совпадающим началом контекста**, и не пересчитывает его заново;
- в ответе приходит `precached_prompt_tokens` - сколько токенов взяли из кэша. Эти токены не тарифицируются;
- библиотека `langchain-gigachat` отдает это число как `usage_metadata["input_token_details"]["cache_read"]`.

Ключевое слово - **начало контекста**. Кэшируется общий префикс, посимвольно совпадающий с прошлым запросом. Отсюда единственное правило, которое надо запомнить.

## Что ломает кэш: порядок блоков в промпте

Промпт агента обычно собирается из четырех кусков: системная инструкция, описания инструментов, вставленная память, история диалога. Первые два неизменны, последние два растут с каждым ходом.

Если поставить память в начало, любое ее обновление меняет весь текст после нее - и совпадающего начала почти не остается. Посмотрим на длину общего префикса между двумя соседними ходами при двух раскладках.

In [6]:
def assemble(system, tools, memory, history, memory_first):
    """Собирает промпт двумя способами: изменчивый блок в начале или в конце."""
    parts = [system, tools]
    parts = ([memory] + parts) if memory_first else (parts + [memory])
    return "\n".join(parts + [history])

SYSTEM_TXT = "Ты ассистент поддержки облачного сервиса. Отвечай кратко и по делу." * 6
TOOLS_TXT  = "Инструменты: check_quota(project), raise_quota(project, limit), check_cluster()." * 12

# ход N и ход N+1: память пополнилась одной записью, история - одной репликой
mem_1 = "Память: тариф Pro; часовой пояс Владивосток."
mem_2 = "Память: тариф Pro; часовой пояс Владивосток; просит писать коротко."
hist_1 = "Клиент: что там по загрузке?"
hist_2 = hist_1 + "\nКлиент: ну и что в итоге с воркерами?"

def common_prefix(a, b):
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i

for memory_first in (True, False):
    p1 = assemble(SYSTEM_TXT, TOOLS_TXT, mem_1, hist_1, memory_first)
    p2 = assemble(SYSTEM_TXT, TOOLS_TXT, mem_2, hist_2, memory_first)
    where = "память в начале" if memory_first else "память в конце"
    print(f"{where:18} | общий префикс: {common_prefix(p1, p2):5} знаков из {len(p2)}")

память в начале    | общий префикс:    43 знаков из 1498
память в конце     | общий префикс:  1407 знаков из 1498


**Пояснения к результату:**
- при памяти в начале общий префикс - несколько десятков знаков: кэшировать нечего, платим за весь промпт заново;
- при памяти в конце неизменными остаются системная инструкция и описания инструментов целиком.

Правило: **сначала неизменное, потом изменчивое.** Системная инструкция и описания инструментов - вперед, память и история - в хвост. Это не микрооптимизация: на длинных описаниях инструментов разница видна в счете.

Теперь измерим попадание на живой модели. Один и тот же длинный префикс уходит дважды с одинаковым `X-Session-ID`.

In [7]:
import uuid
from gigachat.context import session_id_cvar
from langchain_core.messages import SystemMessage, HumanMessage

# X-Session-ID придумываем сами. В проде это идентификатор сессии клиента,
# здесь берем свежий - иначе в кэш попадет уже первый запрос и разницы не увидим.
session_id_cvar.set(f"lesson5-cache-{uuid.uuid4().hex[:8]}")

long_prefix = SYSTEM_TXT + "\n" + TOOLS_TXT

def ask(question):
    resp = llm.invoke([SystemMessage(long_prefix), HumanMessage(question)])
    u = resp.usage_metadata
    return u["input_tokens"], u["input_token_details"].get("cache_read", 0)

first_in, first_cached = ask("Назови один инструмент из списка.")
second_in, second_cached = ask("Назови другой инструмент из списка.")

print(f"первый запрос:  вход {first_in:5} токенов, из кэша {first_cached}")
print(f"второй запрос:  вход {second_in:5} токенов, из кэша {second_cached}")
print(f"сэкономлено во втором запросе: {second_cached} токенов")

первый запрос:  вход   356 токенов, из кэша 4
второй запрос:  вход    21 токенов, из кэша 339
сэкономлено во втором запросе: 339 токенов


**Пояснения к результату:**
- во втором запросе почти весь префикс пришел из кэша, и обратите внимание на `input_tokens`: **кэшированные токены в него не входят**. То есть экономия видна прямо в счетчике входа, а не только в отдельном поле;
- на этом коротком примере разница уже почти десятикратная. На реальном промпте агента с описаниями инструментов она больше.

Если во втором запросе `cache_read` остался нулевым - проверьте три вещи: идентификатор сессии совпадает, префикс совпадает посимвольно, между запросами прошло немного времени. Срок жизни кэша провайдер не обещает фиксированным, и рассчитывать на попадание как на гарантию нельзя: это способ сэкономить, а не способ помнить.

**Здесь и проходит граница темы.** Кэш экономит деньги на том, что мы **и так** тащим в промпте. Он ничего не решает про то, **что** туда класть, и не спасает, когда сессия закончилась. Как только нужно пережить границу сессии - нужна запись наружу.

## Срок жизни записи

Прежде чем писать наружу, стоит решить, когда записанное протухнет. Это дешевле, чем потом разбирать хранилище руками.

Опора простая - **от чего зависит факт**:

| Что записываем | Срок жизни | Почему |
|---|---|---|
| «Часовой пояс Владивосток» | бессрочно | никогда не поменяется |
| «Тариф Pro до декабря» | до декабря | в записи прямо назван срок |
| «Сейчас чинит инцидент с воркерами» | часы | завтра это уже неактуально |
| «Просит писать коротко» | до явной отмены | предпочтение живет, пока его не отменили |

Запись без срока жизни - это обещание хранить ее вечно. Через полгода агент будет уверенно вспоминать клиенту его прошлогодний тариф. Ниже увидим, что в Mem0 срок задается прямо в API.

## Сохранение опыта: свой слой памяти

Прежде чем брать готовую библиотеку, соберем маленькую свою. Не ради упражнения: без нее непонятно, за что именно мы платим Mem0 - а платить придется, и ниже мы посчитаем сколько.

Слой памяти - это три операции: **извлечь** из диалога то, что стоит помнить; **сохранить**; **подставить** в следующий промпт. Начнем с извлечения. Просим модель вернуть записи в фиксированном виде.

In [8]:
import json

EXTRACT_PROMPT = """Ты выделяешь из реплики клиента факты, которые стоит помнить дальше.
Верни только JSON: {"facts": [{"kind": "semantic|episodic|procedural", "text": "..."}]}
Правила:
- факт формулируй самодостаточно, без местоимений;
- предпочтения клиента - semantic, случившиеся события - episodic, порядок действий - procedural;
- если помнить нечего, верни пустой список."""

def extract_facts(turn_text):
    resp = llm.invoke([SystemMessage(EXTRACT_PROMPT), HumanMessage(turn_text)])
    raw = resp.content.strip()
    if raw.startswith("```"):                     # модель любит обертку в блок кода
        raw = raw.split("```")[1].removeprefix("json").strip()
    try:
        return json.loads(raw).get("facts", [])
    except json.JSONDecodeError:
        print("не разобрали ответ модели:", raw[:200])
        return []

for reply in ["Тариф Pro, оплачен до декабря.",
              "И еще: пишите мне без лишних вежливых оборотов, по делу.",
              "Спасибо, помогло!"]:
    print(reply, "->", extract_facts(reply))

Тариф Pro, оплачен до декабря. -> [{'kind': 'semantic', 'text': 'Клиент выбрал тариф Pro.'}, {'kind': 'episodic', 'text': 'Оплата тарифа произведена до декабря.'}]
И еще: пишите мне без лишних вежливых оборотов, по делу. -> [{'kind': 'semantic', 'text': 'Клиент предпочитает общение без лишних вежливых оборотов.'}]
Спасибо, помогло! -> []


Обратите внимание на две строчки, которые в примерах из документации обычно опускают: снятие обертки ```` ``` ```` и `try/except` вокруг разбора JSON. Модель отвечает текстом, а мы ждем структуру - на достаточном числе вызовов она обязательно ответит не так, как мы ждем. Ниже увидим, что в Mem0 ровно эта проблема никуда не делась.

Дальше - хранение и подстановка. Здесь модель не нужна, поэтому запишем факты руками и посмотрим на механику.

In [9]:
from datetime import date, timedelta

STORE = []   # в реальном проекте - таблица или векторный индекс

def remember(kind, text, subject, ttl_days=None):
    STORE.append({
        "kind": kind, "text": text, "subject": subject,
        "created": date(2026, 8, 12),
        "expires": date(2026, 8, 12) + timedelta(days=ttl_days) if ttl_days else None,
    })

remember("semantic",  "Тариф Pro, оплачен до декабря",            "client:42", ttl_days=120)
remember("semantic",  "Часовой пояс Владивосток",                 "client:42")
remember("semantic",  "Просит писать коротко, без вежливых оборотов", "client:42")
remember("episodic",  "12.08 падение воркеров, помогло повышение квоты", "client:42")
remember("semantic",  "Тариф Free",                               "client:42", ttl_days=120)   # устаревшая запись

def recall(subject, today, limit=3):
    live = [r for r in STORE
            if r["subject"] == subject and (r["expires"] is None or r["expires"] >= today)]
    live.sort(key=lambda r: r["created"], reverse=True)
    return live[:limit]

picked = recall("client:42", today=date(2026, 8, 13))
print("подставляем в промпт:")
for r in picked:
    print("  -", r["text"])
print("\nвсего в хранилище:", len(STORE), "| подставили:", len(picked))

подставляем в промпт:
  - Тариф Pro, оплачен до декабря
  - Часовой пояс Владивосток
  - Просит писать коротко, без вежливых оборотов

всего в хранилище: 5 | подставили: 3


Отобрали - теперь надо положить это в промпт, и тут работает правило про порядок блоков: память идет **после** неизменной части.

In [10]:
def build_prompt(system, tools, picked, history):
    memory_block = "Что известно о клиенте:\n" + "\n".join("- " + r["text"] for r in picked)
    # неизменное вперед, изменчивое в хвост - чтобы кэш префикса попадал
    return "\n\n".join([system, tools, memory_block, history])

prompt = build_prompt(
    "Ты ассистент поддержки облачного сервиса. Отвечай кратко и по делу.",
    "Инструменты: check_quota(project), raise_quota(project, limit), check_cluster().",
    picked,
    "Клиент: ну и что в итоге с моими воркерами?",
)
print(prompt)
print("\n---")
print("длина промпта:", len(prompt), "знаков; из них память:",
      sum(len(r["text"]) + 3 for r in picked), "знаков")

Ты ассистент поддержки облачного сервиса. Отвечай кратко и по делу.

Инструменты: check_quota(project), raise_quota(project, limit), check_cluster().

Что известно о клиенте:
- Тариф Pro, оплачен до декабря
- Часовой пояс Владивосток
- Просит писать коротко, без вежливых оборотов

Клиент: ну и что в итоге с моими воркерами?

---
длина промпта: 325 знаков; из них память: 106 знаков


**Пояснения к результату:**
- блок памяти - это обычный текст в промпте: никакой магии, агент видит ровно то, что мы туда положили;
- поэтому подстановка стоит токенов на **каждом** ходу диалога. Три записи стоят копейки, тридцать - уже нет: это и есть причина, по которой при чтении ставят потолок и порог;
- отбор идет по владельцу и по сроку жизни, а в промпт уходит только верхушка - бюджет контекста не резиновый;
- отбор по дате создания - самый грубый из возможных. Настоящий отбор идет по смыслу запроса: эмбеддинги и векторный поиск, это занятие 7;
- в хранилище лежат **две противоречащие записи** про тариф - `Pro` и `Free`, - и посмотрите, что попало в промпт. Прошла одна из них, и выбрал ее не смысл, а порядок вставки: даты создания у обеих одинаковые. В другой день пройдет другая, и агент назовет клиенту другой тариф.

Вот три места, где такой слой ломается, и они же - причина брать готовую библиотеку:

1. **Что считать фактом.** «Спасибо, помогло» помнить не надо, а «пишите коротко» надо. Границу проводит промпт извлечения, и он быстро обрастает исключениями.
2. **Что делать с изменившимся фактом.** Тариф сменился - старую запись надо обновить или удалить, а не положить рядом. Это разрешение противоречий, ему посвящено занятие 8.
3. **Что подставлять, когда записей стало двести.** Нужен отбор по смыслу и порог отсечения.

Mem0 закрывает первое и третье и частично второе. Посмотрим, как - и по какой цене.

## Mem0: слои памяти и что изменилось в версии 2

**Mem0** - библиотека, которая берет на себя весь слой памяти: извлечение записей из диалога, хранение в векторной базе, поиск по смыслу, журнал изменений.

Здесь нужна оговорка, из-за которой ломается половина найденного в интернете кода. В статье Mem0 и во всех туториалах описана **двухфазная** схема: сначала модель извлекает факты, потом вторым вызовом сравнивает их с уже сохраненными и решает - `ADD`, `UPDATE`, `DELETE` или ничего.

**В версии 2 это один вызов.** Модель получает разом новые сообщения, недавно извлеченные записи и похожие существующие - и сразу возвращает `{"memory": [{"text": ..., "event": "ADD"}]}`. Если ваш код ждет старый формат с ключом `facts`, `add()` молча вернет пустой список: ошибки не будет, памяти тоже.

Отсюда практическое правило: **версию Mem0 в проекте закрепляйте жестко** и перечитывайте промпты в исходниках, а не в статье. Как это делается - прямо сейчас.

## Сколько стоит одна запись в память

Заглянем в промпт, которым Mem0 извлекает записи. Он лежит прямо в пакете, читать исходники тут быстрее, чем документацию.

In [11]:
from mem0.configs.prompts import ADDITIVE_EXTRACTION_PROMPT

chars = len(ADDITIVE_EXTRACTION_PROMPT)
print("длина промпта извлечения:", chars, "знаков")
print("язык: английский, первые слова -", ADDITIVE_EXTRACTION_PROMPT.strip()[:34].replace("\n", " "))

# грубая оценка: для английского текста примерно 4 знака на токен
tokens = chars // 4
print("\nпримерно токенов на входе:", tokens)

PRICE_PER_1K = 0.65      # рублей за 1000 токенов, тариф GigaChat плоский: вход примерно равен выходу
one_call = tokens / 1000 * PRICE_PER_1K
print(f"стоимость одного вызова add(): около {one_call:.1f} руб")
print(f"диалог из 20 ходов с записью на каждом ходу: около {one_call * 20:.0f} руб")
print(f"1000 таких диалогов в день: около {one_call * 20 * 1000 / 1000:.0f} тыс. руб в день")

длина промпта извлечения: 33653 знаков
язык: английский, первые слова - # ROLE  You are a Memory Extractor

примерно токенов на входе: 8413
стоимость одного вызова add(): около 5.5 руб
диалог из 20 ходов с записью на каждом ходу: около 109 руб
1000 таких диалогов в день: около 109 тыс. руб в день


**Пояснения к результату:**
- промпт извлечения уходит модели **на каждый** вызов `add()` целиком. Это не разовая настройка, это постоянный расход;
- оценка «четыре знака на токен» грубая, точное число даст токенайзер провайдера. Порядок величины она передает верно;
- заменить этот промпт своим коротким штатными средствами **нельзя**: параметр `prompt=` в `add()` и `custom_instructions` в конфиге не заменяют его, а дописываются к нему - становится только длиннее.

Что с этим делать - три рабочих хода:

1. **Писать реже.** Не на каждом ходу, а в конце сессии или по явному признаку («клиент назвал новый факт»). Один вызов на диалог вместо двадцати.
2. **Не звать модель там, где нечего извлекать.** Готовый эпизод («обращение закрыто») кладется как есть, с `infer=False` - модель не вызывается вовсе.
3. **Кэш контекста.** Промпт на 33 тысячи знаков неизменен от вызова к вызову - это идеальный кандидат на кэширование по `X-Session-ID`. Ровно тот прием, который мы разобрали выше, и здесь он окупается лучше всего.

Это и есть ответ на вопрос из образовательных результатов - когда кэшировать контекст, а когда сохранять опыт. Не «или-или»: сохранять опыт нужно тогда, когда знание должно пережить сессию, а кэш - это то, чем вы сбиваете цену этого сохранения.

## Подключаем Mem0 к GigaChat

Mem0 по умолчанию ходит в OpenAI. Чтобы она работала на GigaChat, обе роли - модель и эмбеддинги - подменяются провайдером `langchain`: туда отдаются готовые объекты `GigaChat` и `GigaChatEmbeddings`.

Три места, где обычно спотыкаются, отмечены в коде.

In [12]:
from mem0 import Memory
from qdrant_client import QdrantClient

# (1) размерность вектора спрашиваем у самих эмбеддингов, а не берем из документации
dims = len(emb.embed_query("проба"))
print("размерность эмбеддингов GigaChat:", dims)

# (2) векторное хранилище держим в оперативной памяти сессии.
#     Если задать path=..., второй вызов Memory.from_config в этом же ноутбуке упадет:
#     Mem0 держит служебную коллекцию в ~/.mem0 мимо вашего конфига.
# (3) промпты Mem0 английские, и на русском диалоге она пишет записи по-английски.
#     Лечится дописыванием к промпту через custom_instructions.
RU_INSTRUCTION = (
    "ВАЖНО: все записи памяти формулируй ТОЛЬКО на русском языке, "
    "даже если рассуждаешь по-английски. Пиши от третьего лица про клиента."
)

config = {
    "llm":      {"provider": "langchain", "config": {"model": llm}},
    "embedder": {"provider": "langchain", "config": {"model": emb}},
    "vector_store": {"provider": "qdrant", "config": {
        "collection_name": "support_agent",
        "client": QdrantClient(":memory:"),
        "embedding_model_dims": dims,
    }},
    "custom_instructions": RU_INSTRUCTION,
}

memory = Memory.from_config(config)
print("Mem0 подключена к GigaChat")

/home/alex/git/ML/Courses/Sber_Advanced_AI_Agents/.venv/lib/python3.13/site-packages/mem0/vector_stores/qdrant.py:177: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(
[PostHog] Multiple active PostHog clients detected for the same project API key and host. Reuse one Posthog instance per app or process when possible to avoid competing background queues and missed shutdown flushes. Multiple clients are supported when intentional.


размерность эмбеддингов GigaChat: 1024
Mem0 подключена к GigaChat


Про третье место стоит сказать подробнее, потому что без него память получается англоязычной. Промпт извлечения у Mem0 написан по-английски, и на русском диалоге модель по инерции пишет записи тоже по-английски - в память попадает `User has paid for the Pro tariff until December`. Дальше этот английский текст подставляется в русский промпт агента.

`custom_instructions` дописывается в конец промпта извлечения (заменить его, напомним, нельзя) - и этого хватает, чтобы записи стали русскими. Тот же текст можно передать разово в `add(..., prompt=...)`.

И четвертое место, невидимое в коде совсем. Своим родным провайдерам Mem0 передает `response_format={"type": "json_object"}` - строгий режим, в котором модель физически не может ответить не-JSON. **Обертка `langchain` этот параметр не передает** - она про него не знает. Значит, на GigaChat формат ответа держится только на тексте промпта. На коротких диалогах разбор устойчив, но гарантии у вас нет: считайте долю неудачных разборов на своем потоке и оборачивайте `add()` повторной попыткой, как мы это делали в своем слое.

## Точки записи: что кладем и как размечаем

Теперь главное - **где в цикле агента стоят вызовы памяти**. У нашего ассистента их три:

| Точка в цикле | Что пишем | Как |
|---|---|---|
| после реплики клиента | факты о клиенте | `add(диалог, infer=True)` - модель извлекает сама |
| после закрытия обращения | эпизод | `add(текст, infer=False)` - кладем как есть, без вызова модели |
| после удачной цепочки инструментов | способ решения | `add(..., metadata={"kind": "procedural"})` |

Разметка типа - это `metadata`. Она не украшение: по ней потом идет отбор при чтении и чистка при устаревании.

In [13]:
# 1. Факты о клиенте: модель извлекает их из диалога сама
turns = [
    {"role": "user", "content": "Тариф Pro, оплачен до декабря. Часовой пояс Владивосток."},
    {"role": "assistant", "content": "Принято."},
    {"role": "user", "content": "И пишите коротко, без вежливых оборотов."},
]
res = memory.add(turns, user_id="client:42", metadata={"kind": "semantic"})
print("извлечено из диалога:", len(res["results"]))
for r in res["results"]:
    print(f'   {r["event"]:7} | {r["memory"]}')

# 2. Готовый эпизод: модель не нужна, кладем как есть
ep = memory.add("12.08 падение воркеров у PRJ-77, помогло повышение квоты",
                user_id="client:42", infer=False, metadata={"kind": "episodic"})
print("\nэпизод записан без вызова модели:", len(ep["results"]))

# 3. Способ решения: он относится не к клиенту, а к типу поломки
pb = memory.add("При падении воркеров с ошибкой квоты: check_quota, затем raise_quota",
                user_id="playbook", infer=False, metadata={"kind": "procedural"})
print("способ решения записан в общий набор:", len(pb["results"]))

print("\nв памяти клиента:", len(memory.get_all(filters={"user_id": "client:42"})["results"]),
      "| в общем наборе:", len(memory.get_all(filters={"user_id": "playbook"})["results"]))

Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"
Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
[PostHog] [FEATURE FLAGS] Unable to evaluate flags remotely: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Read timed out. (read timeout=0.5)
Traceback (most recent call last):
  File "/home/alex/git/ML/Courses/Sber_Advanced_AI_Agents/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py", line 464, in _make_request
    self._validate_conn(conn)
    ~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/home/alex/git/ML/Courses/Sber_Advanced_AI_Agents/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py", line 1106, in _validate_conn
    conn.connect()
    ~~~~~~~~~~~~^^
  File "/home/alex/git/ML/Courses/Sber_Advanced_AI_Agents/.venv/lib/python3.13/site-packages/urllib3/connection.py", l

извлечено из диалога: 3
   ADD     | Клиент оплатил тариф Pro до декабря 2026 года.
   ADD     | Часовой пояс клиента — Владивосток.
   ADD     | Клиент предпочитает короткие сообщения без вежливых оборотов.

эпизод записан без вызова модели: 1
способ решения записан в общий набор: 1

в памяти клиента: 4 | в общем наборе: 1


Обратите внимание на `user_id="playbook"` у третьей записи. Способ решения не принадлежит клиенту - он общий, и класть его в память конкретного пользователя было бы ошибкой: другой клиент с той же поломкой его не найдет.

Библиотека знает и про сами типы: в ней есть перечисление `MemoryType` со значениями `semantic_memory`, `episodic_memory`, `procedural_memory`. Убедимся, что это не выдумка занятия, а штатная вещь.

In [15]:
from mem0.configs.enums import MemoryType

print("типы памяти, которые Mem0 знает штатно:")
for t in MemoryType:
    print("  -", t.value)

типы памяти, которые Mem0 знает штатно:
  - semantic_memory
  - episodic_memory
  - procedural_memory


## Процедурная память: способ решения из траектории

Вернемся к третьему отказу из начала занятия - тому, где агент второй раз подряд перебирает те же тупиковые ходы. Он остался не закрыт: факты о клиенте мы писать научились, а способ решения - нет.

Взять его неоткуда, кроме как из траектории удачного прогона. Правило простое: **берем только те вызовы, которые продвинули решение**, и выбрасываем тупиковые.

In [14]:
def distill_playbook(problem, trajectory):
    """Превращает журнал прогона в короткую запись «как решать такую поломку»."""
    useful = [s for s in trajectory if s["helped"]]
    if not useful:
        return None
    steps = " -> ".join(s["tool"] for s in useful)
    dead_ends = [s["tool"] for s in trajectory if not s["helped"]]
    return {
        "text": f"{problem}: {steps}",
        "kind": "procedural",
        "skip": dead_ends,          # что не помогло - тоже знание
        "confirmed": 1,             # сколько раз способ сработал
    }

YESTERDAY_FULL = [
    {"tool": "restart_workers", "helped": False},
    {"tool": "check_cluster",   "helped": False},
    {"tool": "check_billing",   "helped": False},
    {"tool": "check_quota",     "helped": True},
    {"tool": "raise_quota",     "helped": True},
]

playbook = distill_playbook("падение воркеров с ошибкой квоты", YESTERDAY_FULL)
print("запись:", playbook["text"])
print("не помогло (пропускать):", playbook["skip"])
print()
print("шагов в прогоне:", len(YESTERDAY_FULL), "| шагов в записи:", playbook["text"].count("->") + 1)

запись: падение воркеров с ошибкой квоты: check_quota -> raise_quota
не помогло (пропускать): ['restart_workers', 'check_cluster', 'check_billing']

шагов в прогоне: 5 | шагов в записи: 2


**Пояснения к результату:**
- запись получилась короче прогона: пять шагов сжались до двух. В этом и смысл - в следующий раз агент не повторяет перебор;
- список `skip` не менее ценен, чем сам порядок: «перезапуск воркеров тут не помогает» экономит шаг и время клиента;
- поле `confirmed` - счетчик подтверждений. Способ, сработавший один раз, может быть совпадением; сработавший пять раз - уже правило. Повышать счетчик при каждом удачном применении дешево, а доверие к записи получается измеримым.

Признак `helped` в журнале не появляется сам - его проставляет либо код (вызов вернул решение), либо модель при разборе прогона. Это и есть цена процедурной памяти: сама по себе она не заводится, нужен разбор удачных прогонов.

Записывается такая запись туда же, но в общий набор, а не в память клиента.

In [ ]:
memory.add(playbook["text"], user_id="playbook", infer=False,
           metadata={"kind": "procedural", "skip": ", ".join(playbook["skip"]),
                     "confirmed": playbook["confirmed"]})

found = memory.search("воркеры падают из-за квоты", filters={"user_id": "playbook"},
                      top_k=2, threshold=0.3)["results"]
for r in found:
    print(r["memory"], "| не помогает:", (r.get("metadata") or {}).get("skip"))

## Чтение памяти: порог и объяснение

Читать всю память в промпт нельзя - ради этого все и затевалось. Читаем по смыслу запроса, с порогом и потолком.

In [16]:
found = memory.search(
    "какой у клиента тариф",
    filters={"user_id": "client:42"},
    top_k=3,          # потолок: сколько записей вообще берем
    threshold=0.3,    # порог: ниже него запись считается нерелевантной
    explain=True,     # положить разбор оценки в поле score_details
)

for r in found["results"]:
    print(f'{r["score"]:.3f} | {r["memory"]} | тип: {(r.get("metadata") or {}).get("kind")}')
    if r.get("score_details"):
        print("        из чего сложилась оценка:", r["score_details"])

0.845 | Клиент оплатил тариф Pro до декабря 2026 года. | тип: semantic
        из чего сложилась оценка: {'semantic_score': 0.8450240431004505, 'bm25_score': 0.0, 'entity_boost': 0.0, 'raw_score': 0.8450240431004505, 'max_possible_score': 1.0, 'final_score': 0.8450240431004505, 'threshold': 0.3}
0.821 | Часовой пояс клиента — Владивосток. | тип: semantic
        из чего сложилась оценка: {'semantic_score': 0.8212857994086269, 'bm25_score': 0.0, 'entity_boost': 0.0, 'raw_score': 0.8212857994086269, 'max_possible_score': 1.0, 'final_score': 0.8212857994086269, 'threshold': 0.3}
0.804 | Клиент предпочитает короткие сообщения без вежливых оборотов. | тип: semantic
        из чего сложилась оценка: {'semantic_score': 0.8041394548083179, 'bm25_score': 0.0, 'entity_boost': 0.0, 'raw_score': 0.8041394548083179, 'max_possible_score': 1.0, 'final_score': 0.8041394548083179, 'threshold': 0.3}


Порог `0.3` здесь стоит не от балды - это значение по умолчанию в Mem0. Проверим, что он на самом деле отсекает. Возьмем несколько записей и три запроса, последний из которых заведомо не про клиента.

In [17]:
import numpy as np

RECORDS = [
    "Тариф Pro, оплачен до декабря",
    "Часовой пояс Владивосток",
    "Просит писать коротко, без вежливых оборотов",
    "12.08 падение воркеров у PRJ-77, помогло повышение квоты",
    "Столица Франции - Париж",
]

vecs = np.array(emb.embed_documents(RECORDS))
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

for query in ["какой у клиента тариф", "во сколько ему удобно звонить", "рецепт борща"]:
    q = np.array(emb.embed_query(query))
    q = q / np.linalg.norm(q)
    sims = sorted(zip(RECORDS, vecs @ q), key=lambda x: -x[1])
    print(f"запрос: «{query}»")
    for rec, s in sims:
        print(f"   {s:.3f} | {rec}")
    print(f"   разрыв между первым и вторым: {sims[0][1] - sims[1][1]:.3f}\n")

запрос: «какой у клиента тариф»
   0.849 | Тариф Pro, оплачен до декабря
   0.796 | Просит писать коротко, без вежливых оборотов
   0.762 | 12.08 падение воркеров у PRJ-77, помогло повышение квоты
   0.758 | Столица Франции - Париж
   0.731 | Часовой пояс Владивосток
   разрыв между первым и вторым: 0.053

запрос: «во сколько ему удобно звонить»
   0.813 | Просит писать коротко, без вежливых оборотов
   0.798 | Тариф Pro, оплачен до декабря
   0.778 | Часовой пояс Владивосток
   0.770 | 12.08 падение воркеров у PRJ-77, помогло повышение квоты
   0.755 | Столица Франции - Париж
   разрыв между первым и вторым: 0.016

запрос: «рецепт борща»
   0.803 | Просит писать коротко, без вежливых оборотов
   0.781 | Тариф Pro, оплачен до декабря
   0.759 | Столица Франции - Париж
   0.756 | 12.08 падение воркеров у PRJ-77, помогло повышение квоты
   0.735 | Часовой пояс Владивосток
   разрыв между первым и вторым: 0.021



**Пояснения к результату:**
- все близости лежат в узком коридоре примерно от 0.73 до 0.85. **Порог 0.3 не отсекает вообще ничего** - под него не проходит только полностью чужой текст, которого в памяти клиента и не бывает;
- на запросе «рецепт борща», не имеющем к памяти никакого отношения, лучшая запись все равно набирает больше 0.80. Абсолютный порог, взятый из документации, тут бесполезен;
- работает другое: **порог, посчитанный по своим данным** (посмотрите на распределение, как в этой ячейке, и возьмите значение выше основной массы), и **разрыв между первой записью и второй**. Для осмысленного запроса разрыв заметный, для чужого - его нет;
- и самый дешевый прием: всегда прогонять заведомо посторонний запрос. Если он что-то возвращает выше порога, порог занижен.

Это свойство конкретных эмбеддингов, а не Mem0: у другой модели коридор будет другим. Поэтому первое, что стоит сделать на своих данных, - построить вот такую табличку.

Как выбирать порог по-взрослому - занятие 8.

Два поля в выдаче стоит смотреть не только при отладке. `score` есть всегда, а `score_details` появляется при `explain=True` и показывает, из чего сложилась оценка, - самый прямой способ понять, почему агент вспомнил не то. И обратите внимание: разметка `metadata`, которую мы проставили при записи, доезжает до чтения в целости - по ней можно отбирать записи нужного типа.

## Память внутри цикла агента

До сих пор мы дергали `add()` и `search()` по отдельности. В работающем агенте они стоят в конкретных местах цикла, и от расстановки зависит и качество, и счет.

Порядок такой: **прочитать перед вызовом модели, ответить, записать после хода**. Собираем это целиком.

In [18]:
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM = "Ты ассистент поддержки облачного сервиса. Отвечай кратко и по делу."
TOOLS_DESC = "Инструменты: check_quota(project), raise_quota(project, limit), check_cluster()."

def recall_block(user_id, question, top_k=3, threshold=0.3):
    """Точка чтения: что подставить в промпт под этот конкретный вопрос."""
    found = memory.search(question, filters={"user_id": user_id},
                          top_k=top_k, threshold=threshold)["results"]
    if not found:
        return "", []
    lines = "\n".join("- " + r["memory"] for r in found)
    return f"Что известно о клиенте:\n{lines}", found

def answer(user_id, question, write_back=True):
    """Один ход диалога: прочитали память -> ответили -> записали новое."""
    mem_block, used = recall_block(user_id, question)
    # Системное сообщение у GigaChat может быть только одно и только первым,
    # поэтому память дописываем в его конец, а не отдельным SystemMessage.
    # Заодно это и есть нужный порядок: неизменное вперед, изменчивое в хвост.
    system = SYSTEM + "\n" + TOOLS_DESC + ("\n\n" + mem_block if mem_block else "")
    messages = [SystemMessage(system), HumanMessage(question)]

    reply = llm.invoke(messages).content

    written = []
    if write_back:   # точка записи: после хода, а не до
        res = memory.add([{"role": "user", "content": question},
                          {"role": "assistant", "content": reply}],
                         user_id=user_id, metadata={"kind": "semantic"})
        written = [r["memory"] for r in res["results"]]
    return reply, [r["memory"] for r in used], written

reply, used, written = answer("client:42", "Напомни, какой у меня тариф и до какого числа оплачен")
print("ответ агента :", reply)
print("прочитано из памяти:", used)
print("записано в память  :", written)

ответ агента : У вас тариф Pro, оплата действительна до декабря 2026 года.
прочитано из памяти: ['Клиент оплатил тариф Pro до декабря 2026 года.', 'Часовой пояс клиента — Владивосток.', 'Клиент предпочитает короткие сообщения без вежливых оборотов.']
записано в память  : ['У клиента тариф Pro, оплаченный до декабря 2026 года.']


Первое, обо что тут спотыкаются: **у GigaChat системное сообщение может быть только одно и только первым**. Привычный по примерам LangChain прием «положим память отдельным `SystemMessage`» дает ошибку `400: system message must be the first message`. Поэтому память дописывается в конец единственного системного сообщения - что удобно совпадает с правилом про порядок блоков.

Дальше три вещи, которые видно в этом коде и не видно в примерах из документации:

- **чтение идет под конкретный вопрос**, а не «весь профиль клиента». Иначе бюджет контекста съедается памятью;
- **запись идет после ответа**, а не до. Записав до, вы подставите в тот же промпт то, что клиент только что сказал, - двойная оплата за один и тот же текст;
- **запись можно выключить** (`write_back=False`). Это не тумблер для красоты: ниже посчитаем, что он экономит.

## Писать на каждом ходу или в конце сессии

Самое дорогое решение в этом занятии - как часто звать `add()`. Посчитаем оба варианта на диалоге из двенадцати ходов.

In [20]:
EXTRACT_TOKENS = 8413        # промпт извлечения Mem0, посчитали выше
DIALOG_TOKENS  = 350         # средний ход диалога: промпт агента плюс ответ
PRICE_PER_1K   = 0.65
TURNS          = 12

def cost(tokens):
    return tokens / 1000 * PRICE_PER_1K

talk = cost(DIALOG_TOKENS * TURNS)
every_turn = cost((EXTRACT_TOKENS + DIALOG_TOKENS) * TURNS)          # add() на каждом ходу
session_end = cost(DIALOG_TOKENS * TURNS + EXTRACT_TOKENS)           # add() один раз в конце
cached = cost(DIALOG_TOKENS * TURNS + EXTRACT_TOKENS * TURNS * 0.1)  # кэш префикса, платим ~10 процентов

print(f"диалог без памяти вообще      : {talk:7.1f} руб")
print(f"память на каждом ходу         : {every_turn:7.1f} руб  (x{every_turn/talk:.0f} к диалогу)")
print(f"память один раз в конце сессии: {session_end:7.1f} руб  (x{session_end/talk:.1f})")
print(f"на каждом ходу, но с кэшем    : {cached:7.1f} руб  (x{cached/talk:.1f})")

диалог без памяти вообще      :     2.7 руб
память на каждом ходу         :    68.4 руб  (x25 к диалогу)
память один раз в конце сессии:     8.2 руб  (x3.0)
на каждом ходу, но с кэшем    :     9.3 руб  (x3.4)


**Пояснения к результату:**
- запись на каждом ходу дороже самого диалога в несколько раз - память становится основной статьей расхода, а не добавкой;
- перенос записи в конец сессии убирает почти весь перерасход, но платит за это задержкой знания: внутри этого же диалога агент новых фактов не увидит;
- кэш префикса дает средний вариант. Доля попадания в 10 процентов взята для примера - меряйте свою по `precached_prompt_tokens`.

Отсюда рабочее правило: **писать по событию, а не по расписанию.** Клиент назвал новый факт - пишем сразу. Обычный уточняющий ход - копим до конца сессии.

## Срок жизни и журнал изменений

Две вещи, которые в Mem0 есть штатно и которые обычно пишут руками.

**Срок жизни** задается прямо в `add()`. Просроченные записи перестают попадать в выдачу, но не пропадают - их можно посмотреть отдельно.

In [ ]:
# срок ставим вчерашним днем, чтобы прямо сейчас увидеть, как работает отсечение
memory.add("Сейчас чинит инцидент с воркерами", user_id="client:42",
           infer=False, metadata={"kind": "episodic"}, expiration_date="2026-08-12")

live = memory.get_all(filters={"user_id": "client:42"})["results"]
all_rec = memory.get_all(filters={"user_id": "client:42"}, show_expired=True)["results"]
print("видно агенту:", len(live), "| всего в хранилище:", len(all_rec))

# журнал изменений по записи: что с ней происходило
some_id = live[0]["id"]
for h in memory.history(some_id):
    print(f'{h["event"]:7} | было: {h["old_memory"]} -> стало: {h["new_memory"]}')

Журнал (`history`) отвечает на вопрос «откуда агент вообще взял этот факт и когда он изменился». Без него разбор жалобы превращается в гадание: в векторной базе лежит текущее состояние, а не то, как оно таким стало.

Это же основа для удаления по требованию клиента и для аудита записей - тема занятия 8.

## Грабли, на которые вы наступите

Собрано из того, что ломается на практике при первом подключении:

- **Второй `Memory.from_config` в одном процессе падает**, если задан `path`. Mem0 держит служебную коллекцию в `~/.mem0` мимо вашего конфига, и второй клиент не может ее открыть. Закрытие первого клиента не помогает. Обход - готовый `QdrantClient(":memory:")`, как выше.
- **Промпты Mem0 английские, и записи выходят английскими** даже на полностью русском диалоге: в память ложится `User has paid for the Pro tariff until December`. Лечится `custom_instructions`, как выше. Без этого английский текст подставляется в русский промпт агента.
- **Полный пакет `langchain` обязателен.** Без него провайдер `langchain` не импортируется, а сообщение об ошибке говорит про langchain вообще, а не про то, что не хватает конкретного пакета.
- **Строгий JSON-режим через langchain не работает** - про это выше.
- **Дедупликация ведет себя по-разному от прогона к прогону.** Mem0 сравнивает новое с похожими существующими записями, и на одном прогоне она не добавляет ничего («уже знаю»), а на другом, при чуть иной формулировке, заводит вторую запись про то же самое. Как на гарантию полагаться нельзя: разбор противоречий и слияние - занятие 8.
- **API ломается в минорных выпусках.** Закрепляйте версию и перечитывайте исходники при обновлении.

## Mirix: что будет, если довести раскладку по типам до конца

Мы начали занятие с трех делений памяти и сказали, что в реальном агенте заполнены не все клетки. **Mirix** - система, авторы которой пошли в противоположную сторону и разложили память на шесть отдельных видов, у каждого свое хранилище:

- **Core** - устойчивое ядро: кто такой пользователь, кто такой агент, каким тоном общаться;
- **Episodic** - события и переживания пользователя;
- **Semantic** - понятия и сущности: что означает термин, кто такой упомянутый человек;
- **Procedural** - пошаговые инструкции, как что делается;
- **Resource** - документы, файлы, картинки, которыми делился пользователь;
- **Knowledge Vault** - хранилище точных данных: реквизиты, идентификаторы, ключи.

Главное в устройстве - не сами шесть видов, а то, что **у каждого свой управляющий агент**, а над ними стоит мета-менеджер: он решает, в какое хранилище отправить новую информацию и из какого читать под конкретный запрос. То есть маршрутизация записи, которую в Mem0 делает один промпт, здесь вынесена в отдельный слой из семи агентов.

Заявленный результат - на многомодальном наборе (поток снимков экрана) точность выше базового RAG примерно на треть при радикально меньшем объеме хранения: система хранит извлеченный смысл, а не сами кадры.

**Когда это оправдано.** Когда потоков разного рода много и они правда разные: текст, файлы, снимки экрана, реквизиты. Персональный помощник, который наблюдает за работой пользователя, - как раз такой случай.

**Чем платите.** Семь дополнительных агентов - это семь дополнительных вызовов модели на маршрутизацию, отдельная база под каждое хранилище и заметно более сложная отладка: чтобы понять, почему агент чего-то не вспомнил, надо пройти по всей цепочке. Для ассистента поддержки, где вся память - это профиль клиента и десяток эпизодов, такая раскладка не окупается.

## Что выбрать под задачу

| Ситуация | Чего хватит |
|---|---|
| Диалог живет одну сессию, все влезает в окно | ничего не выносить наружу, следить за порядком блоков ради кэша |
| Нужно помнить профиль и предпочтения между сессиями | внешняя запись фактов: свой слой на десятки строк или Mem0 |
| Много записей, нужен отбор по смыслу и журнал | Mem0 |
| Разнородные потоки: файлы, снимки экрана, реквизиты | раскладка по видам, как в Mirix |
| Агент повторяет одни и те же неудачные ходы | процедурная память - и это чаще всего никем не сделано |

И отдельно: **память - не бесплатная добавка**. Каждая запись стоит вызова модели, каждая подстановка - токенов в промпте. Если задача решается окном контекста, память в нее добавлять не надо.

## Чек-лист памяти агента

Перед тем как выкатывать агента с памятью:

- для каждого типа записи назван **владелец**: клиент, задача или общий набор. Способ решения в памяти одного клиента - ошибка;
- у записи есть **срок жизни** или явное решение, что она бессрочна;
- **точки записи** названы поименно (после реплики клиента, после закрытия обращения, после удачной цепочки) - а не «пишем все подряд»;
- на чтении стоят **потолок и порог**, порог посчитан по своим данным и проверен заведомо посторонним запросом;
- промпт собирается **неизменное вперед, изменчивое в хвост**, идентификатор сессии передается - кэш попадает;
- посчитано, **сколько стоит запись** на вашем объеме диалогов, и выбрано, писать по событию или в конце сессии;
- есть способ ответить на вопрос **«откуда агент это взял»**: журнал изменений или свой аудит записей;
- проверено, **на каком языке** легли записи, и что в промпт агента подставляется то, что вы ожидали;
- версия библиотеки памяти закреплена, поведение перепроверено после обновления.

## Практика

Соберите слой памяти для ассистента поддержки на Mem0 с разметкой типов и явными точками записи.

Что нужно сделать:

1. Подключите Mem0 к GigaChat (модель и эмбеддинги - через провайдер `langchain`).
2. Проведите через агента диалог из шести реплик ниже. Расставьте **три точки записи**: факты о клиенте, эпизод по итогу обращения, способ решения.
3. Каждой записи проставьте `metadata={"kind": ...}` - `semantic`, `episodic` или `procedural`, а фактам с ограниченным сроком - `expiration_date`.
4. Прочитайте память обратно: на запрос «какой тариф у клиента» посмотрите, сколько записей прошло порог `0.3`. Их будет несколько, включая явно лишние, - подберите порог по распределению близостей, как мы делали выше. Способ решения при этом должен находиться **не** по идентификатору клиента.

Диалог для прогона - в ячейке ниже. Ожидаемый итог: в памяти клиента три-четыре семантические записи и один эпизод, в общем наборе - одна процедурная.

In [21]:
PRACTICE_DIALOG = [
    {"role": "user", "content": "Добрый день. Проект PRJ-91, тариф Team, оплачен до марта."},
    {"role": "assistant", "content": "Здравствуйте! Слушаю."},
    {"role": "user", "content": "Сборки встают в очередь и не стартуют. Началось час назад."},
    {"role": "assistant", "content": "Проверяю лимит параллельных сборок."},
    {"role": "user", "content": "И отвечайте, пожалуйста, без вводных фраз."},
    {"role": "assistant", "content": "Лимит исчерпан, поднял до 10. Сборки пошли."},
]

# ваш код

**Разбор решения.** Ниже один из рабочих вариантов - сверьтесь с ним после того, как напишете свой.

In [ ]:
# 1. Факты о клиенте: извлекаем моделью из клиентских реплик
client_turns = [t for t in PRACTICE_DIALOG if t["role"] == "user"]
memory.add(client_turns, user_id="client:91", metadata={"kind": "semantic"})

# 2. Факт с ограниченным сроком пишем отдельно, со сроком жизни
memory.add("Тариф Team оплачен до марта 2027", user_id="client:91",
           infer=False, metadata={"kind": "semantic"}, expiration_date="2027-03-01")

# 3. Эпизод по итогу обращения: модель не нужна
memory.add("13.08 сборки PRJ-91 стояли в очереди, помогло повышение лимита параллельных сборок до 10",
           user_id="client:91", infer=False, metadata={"kind": "episodic"})

# 4. Способ решения - в общий набор, не в память клиента
memory.add("Сборки стоят в очереди: проверить лимит параллельных сборок, затем поднять лимит",
           user_id="playbook", infer=False, metadata={"kind": "procedural"})

# 5. Проверка чтением
tariff = memory.search("какой тариф у клиента", filters={"user_id": "client:91"},
                       top_k=3, threshold=0.3)["results"]
play = memory.search("сборки стоят в очереди", filters={"user_id": "playbook"},
                     top_k=1, threshold=0.3)["results"]

print("про тариф найдено:", [r["memory"] for r in tariff])
print("способ решения:   ", [r["memory"] for r in play])

про тариф найдено: ['Тариф Team оплачен до марта 2027', 'Проект PRJ-91 тарифа Team оплачен клиентом до марта 2027 года.', 'Клиент сообщил о проблеме с проектом PRJ-91 на тарифе Team: сборки встают в очередь и не запускаются с 25 августа 2026 года примерно с 13:00.']
способ решения:    ['Сборки стоят в очереди: проверить лимит параллельных сборок, затем поднять лимит']


## Итоги

В этом уроке мы:

1. Разобрали три деления памяти - по типу информации, по длительности и по исполнению - и увидели, где они работают как проверка покрытия, а где ничего не решают.
2. Научились выбирать между кэшем контекста и записью наружу: измерили, как порядок блоков в промпте влияет на попадание в кэш, и посчитали, во сколько обходится сохранение опыта.
3. Подключили Mem0 к агенту на GigaChat, разметили записи по типам, расставили точки записи в цикле и прочитали память с порогом; разобрали устройство Mirix и случаи, когда раскладка на шесть видов оправдана.

## Что запомнить

- «Агент забыл» - это три разных отказа: не влезло в окно, кончилась сессия, опыт нигде не осел. Лечатся они разным, и первый шаг - понять, какой из трех перед вами.
- Классификации памяти не говорят, что делать. Решения принимаются по четырем вопросам: что записать, когда записать, где хранить, что подставить в промпт.
- Кэш контекста и внешняя память - не альтернативы. Кэш сбивает цену того, что вы **и так** передаете; память решает, что вообще передавать.
- Кэшируется совпадающее начало промпта. Отсюда правило сборки: неизменное вперед (инструкция, описания инструментов), изменчивое в хвост (память, история).
- Записи нужен срок жизни. Без него агент через полгода уверенно вспомнит клиенту прошлогодний тариф.
- Память стоит денег: в Mem0 версии 2 промпт извлечения - около 33 тысяч знаков на **каждый** вызов `add()`. Пишите реже, кладите готовые эпизоды с `infer=False`, экономьте на кэше префикса.
- В Mem0 версии 2 извлечение однофазное и возвращает `{"memory": [...]}`, а не `{"facts": [...]}` из статьи. Версию закрепляйте, промпты читайте в исходниках.
- Процедурная память - самая недооцененная. Профиль клиента хранят все, а способ решения - почти никто, и агент раз за разом повторяет одни и те же тупиковые ходы.

## Полезные материалы

- [Документация Mem0](https://docs.mem0.ai/) - конфигурация провайдеров, API `add`/`search`/`history`.
- [Исходники Mem0](https://github.com/mem0ai/mem0) - промпты лежат в `mem0/configs/prompts.py`; при обновлении версии смотреть надо туда.
- [Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory](https://arxiv.org/abs/2504.19413) - статья с двухфазной схемой извлечения; полезна для понимания замысла, но расходится с текущим кодом.
- [MIRIX: Multi-Agent Memory System for LLM-Based Agents](https://arxiv.org/abs/2507.07957) - шесть видов памяти и мета-менеджер маршрутизации.
- [Документация GigaChat: работа с историей чата](https://developers.sber.ru/docs/ru/gigachat/guides/keeping-context) - кэширование контекста, `X-Session-ID`, `precached_prompt_tokens`.
- [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560) - иерархия памяти и вытеснение контекста; подробно разбирается на занятии 6.